# MLflow GenAI — Complete Demo Notebook

This single notebook walks through every major MLflow 3.x GenAI feature
using a simple Python Q&A chatbot as the example application.

| Section | Feature | MLflow UI Tab |
|---|---|---|
| 1 | Experiment tracking | Overview |
| 2 | Tracing + Autolog | Traces |
| 3 | Multi-turn Sessions | Traces → Sessions |
| 4 | LLM Evaluation | Evaluation runs |
| 5 | Evaluation Datasets | Datasets |
| 6 | Judges | Judges |

> **Before running:** start MLflow server
> ```
> mlflow server --host 127.0.0.1 --port 5000
> ```

## Setup

In [ ]:
!pip install mlflow google-genai --quiet

In [ ]:
import os, uuid, time, tempfile
import mlflow
from google import genai
from google.genai import types
from mlflow.genai import scorer
from mlflow.genai.scorers import Correctness, ScorerSamplingConfig
from mlflow.genai.datasets import create_dataset, search_datasets
from mlflow.entities import Feedback

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

EXPERIMENT = "MLflow-GenAI-Demo"
GATEWAY    = "gateway:/gemini-judge"   # AI Gateway endpoint name

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENT)
EXP_ID = mlflow.get_experiment_by_name(EXPERIMENT).experiment_id

print("MLflow", mlflow.__version__)
print("Experiment ID:", EXP_ID)

---
## Section 1 — Experiment Tracking

MLflow tracks every run with params, metrics and artifacts.
Think of a **run** as one experiment attempt — you can compare many runs side by side.

**UI tab:** Overview

In [ ]:
questions = [
    "What is a Python list?",
    "What is a Python dictionary?",
    "What is a Python function?",
]

for temp in [0.0, 0.5, 1.0]:
    with mlflow.start_run(run_name=f"temperature-{temp}"):

        # Log the configuration used for this run
        mlflow.log_param("model",       "gemini-2.5-flash")
        mlflow.log_param("temperature", temp)
        mlflow.log_param("topic",       "python-basics")

        t0       = time.time()
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[questions[0]],
            config=types.GenerateContentConfig(temperature=temp)
        )
        latency = round(time.time() - t0, 3)

        # Log measured values
        mlflow.log_metric("latency_seconds", latency)
        mlflow.log_metric("word_count",      len((response.text or "").split()))

        # Save the response as a text file artifact
        with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
            f.write(response.text or "")
            tmp = f.name
        mlflow.log_artifact(tmp, artifact_path="responses")

        print(f"temp={temp} | latency={latency}s | words={len((response.text or '').split())}")

print("\nOpen Overview tab -> select all 3 runs -> Compare")

---
## Section 2 — Tracing

Tracing records every LLM call as a structured timeline.
You can see exactly what was sent to the model and what came back.

**UI tab:** Traces

In [ ]:
# autolog() traces every Gemini call automatically from this point on
mlflow.gemini.autolog()
print("Autolog ON — every Gemini call will now appear in Traces tab")

In [ ]:
# Method 1: autolog captures this automatically
with mlflow.start_run(run_name="autolog-demo"):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=["Explain what a Python decorator is in one sentence."]
    )
    print(response.text)

In [ ]:
# Method 2: @mlflow.trace decorator — gives the function a named span
@mlflow.trace
def ask_python_question(question: str) -> str:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[question]
    )
    return (response.text or "").strip()


with mlflow.start_run(run_name="manual-trace-demo"):
    answer = ask_python_question("What is a Python lambda function?")
    print(answer)

print("\nOpen Traces tab -> click a trace -> see request/response timeline")

---
## Section 3 — Multi-turn Sessions

A session groups multiple traces from the same conversation.
Each question is one trace. All traces share the same `session_id`
so MLflow groups them together in the Sessions view.

**UI tab:** Traces → Sessions

In [ ]:
# Session 1: student asking Python beginner questions
session_id_1 = str(uuid.uuid4())
print(f"Session 1: {session_id_1[:8]}...")

chat1 = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction="You are a Python tutor. Answer in 2 sentences max."
    )
)

@mlflow.trace
def student_turn_1(user_msg: str) -> str:
    # This links every trace to session_id_1 in the Sessions tab
    mlflow.update_current_trace(session_id=session_id_1)
    response = chat1.send_message(user_msg)
    return (response.text or "").strip()


turns1 = [
    "What is a variable in Python?",
    "What is the difference between a list and a tuple?",
    "How do I write a for loop in Python?"
]

for i, msg in enumerate(turns1, 1):
    answer = student_turn_1(msg)
    print(f"[Turn {i}] Q: {msg}")
    print(f"         A: {answer}\n")

In [ ]:
# Session 2: different student, asking about functions
session_id_2 = str(uuid.uuid4())
print(f"Session 2: {session_id_2[:8]}...")

chat2 = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction="You are a Python tutor. Answer in 2 sentences max."
    )
)

@mlflow.trace
def student_turn_2(user_msg: str) -> str:
    mlflow.update_current_trace(session_id=session_id_2)
    response = chat2.send_message(user_msg)
    return (response.text or "").strip()


turns2 = [
    "What is a Python function?",
    "What does the return statement do?"
]

for i, msg in enumerate(turns2, 1):
    answer = student_turn_2(msg)
    print(f"[Turn {i}] Q: {msg}")
    print(f"         A: {answer}\n")

print("Open Traces -> Sessions tab")
print(f"Session 1 ({session_id_1[:8]}...): 3 turns")
print(f"Session 2 ({session_id_2[:8]}...): 2 turns")

---
## Section 4 — LLM Evaluation

`mlflow.genai.evaluate()` runs your dataset through scorers and logs
every result as an **Evaluation run** — so you can track model quality over time.

**UI tab:** Evaluation runs

In [ ]:
# Evaluation dataset: questions + expected answers
eval_dataset = [
    {
        "inputs":       {"question": "What is a Python list?"},
        "expectations": {"expected_response": "A list is an ordered, mutable collection of items in Python."},
    },
    {
        "inputs":       {"question": "What is a Python dictionary?"},
        "expectations": {"expected_response": "A dictionary stores key-value pairs and allows fast lookup by key."},
    },
    {
        "inputs":       {"question": "What is a Python function?"},
        "expectations": {"expected_response": "A function is a reusable block of code defined with the def keyword."},
    },
]

# Generate model outputs and add them to the dataset
for row in eval_dataset:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[row["inputs"]["question"]],
        config=types.GenerateContentConfig(system_instruction="Answer in 1-2 sentences.")
    )
    row["outputs"] = (response.text or "").strip()

for row in eval_dataset:
    print(f"Q: {row['inputs']['question']}")
    print(f"A: {row['outputs']}\n")

In [ ]:
# Scorer 1: Gemini judges if the answer is factually correct
@scorer
def correctness_judge(inputs, outputs, expectations) -> Feedback:
    question     = inputs.get("question", "")
    answer       = str(outputs or "")
    ground_truth = expectations.get("expected_response", "")
    prompt = (
        "Is this answer factually correct?\n"
        f"Question: {question}\n"
        f"Correct answer: {ground_truth}\n"
        f"Given answer: {answer}\n"
        "Reply: true or false, then one sentence reason."
    )
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[prompt],
        config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=60)
    )
    text = (response.text or "").strip()
    return Feedback(value=text.lower().startswith("true"), rationale=text)


# Scorer 2: Simple code check — no LLM needed
@scorer
def conciseness_judge(outputs) -> Feedback:
    word_count = len(str(outputs or "").split())
    is_concise = word_count <= 30
    label      = "concise" if is_concise else "too long"
    return Feedback(value=is_concise, rationale=f"{word_count} words - {label}")


print("Scorers ready")

In [ ]:
# Run evaluation — results logged to Evaluation runs tab
results = mlflow.genai.evaluate(
    data=eval_dataset,
    scorers=[correctness_judge, conciseness_judge],
)
print("Done — open Evaluation runs tab to see per-row scores and rationale")

---
## Section 5 — Evaluation Datasets

`create_dataset()` registers a versioned dataset visible in the **Datasets tab**.
You can track which dataset version produced which evaluation score.

**UI tab:** Datasets

In [ ]:
# Version 1: 3 basic Python questions
dataset_v1 = create_dataset(
    name="python-qa-v1",
    experiment_id=EXP_ID,
    tags={"version": "v1", "topic": "python-basics"},
)

dataset_v1.merge_records([
    {
        "inputs":       {"question": "What is a Python list?"},
        "expectations": {"expected_response": "A list is an ordered, mutable collection of items."},
    },
    {
        "inputs":       {"question": "What is a Python dictionary?"},
        "expectations": {"expected_response": "A dictionary stores key-value pairs for fast lookup."},
    },
    {
        "inputs":       {"question": "What is a Python function?"},
        "expectations": {"expected_response": "A function is a reusable block of code defined with def."},
    },
])

print(f"Dataset v1 created: {dataset_v1.dataset_id}")
print(f"Records: {len(dataset_v1.to_df())}")

In [ ]:
# Version 2: adds 2 harder questions
dataset_v2 = create_dataset(
    name="python-qa-v2",
    experiment_id=EXP_ID,
    tags={"version": "v2", "topic": "python-intermediate"},
)

dataset_v2.merge_records([
    {
        "inputs":       {"question": "What is a Python list?"},
        "expectations": {"expected_response": "A list is an ordered, mutable collection of items."},
    },
    {
        "inputs":       {"question": "What is a Python dictionary?"},
        "expectations": {"expected_response": "A dictionary stores key-value pairs for fast lookup."},
    },
    {
        "inputs":       {"question": "What is a Python function?"},
        "expectations": {"expected_response": "A function is a reusable block of code defined with def."},
    },
    {
        "inputs":       {"question": "What is a Python decorator?"},
        "expectations": {"expected_response": "A decorator wraps a function to add behaviour without modifying it."},
    },
    {
        "inputs":       {"question": "What is the difference between *args and **kwargs?"},
        "expectations": {"expected_response": "*args passes variable positional arguments, **kwargs passes variable keyword arguments."},
    },
])

print(f"Dataset v2 created: {dataset_v2.dataset_id}")
print(f"Records: {len(dataset_v2.to_df())}")
print("Open Datasets tab — you should see python-qa-v1 and python-qa-v2")

In [ ]:
# Run model against both datasets and log scores
# This shows the score drops on v2 because the questions are harder
def word_overlap(pred, gt):
    p = set(pred.lower().split())
    g = set(gt.lower().split())
    return round(len(p & g) / max(len(g), 1), 3)


for dataset, run_name in [(dataset_v1, "eval-v1"), (dataset_v2, "eval-v2")]:
    df = dataset.to_df()
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("dataset", dataset.name)
        scores = []
        for _, row in df.iterrows():
            q  = row["inputs"]["question"]
            gt = row["expectations"]["expected_response"]
            pred = (client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[q],
                config=types.GenerateContentConfig(system_instruction="Answer in 1-2 sentences.")
            ).text or "").strip()
            scores.append(word_overlap(pred, gt))
        avg = round(sum(scores) / len(scores), 4)
        mlflow.log_metric("avg_word_overlap", avg)
        print(f"{run_name}: avg_word_overlap = {avg}")

---
## Section 6 — Judges

A registered judge runs **automatically on every new trace**.
Once active, every Gemini call in your app gets scored without any extra code.

**UI tab:** Judges

> Requires the AI Gateway endpoint `gemini-judge` configured in the MLflow UI.

In [ ]:
# Register Correctness judge using the AI Gateway endpoint
# After this runs, every new trace gets auto-scored for correctness
correctness = Correctness(model=GATEWAY)
registered  = correctness.register(name="correctness_judge")
registered.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))

print("Judge registered and active")
print("Open the Judges tab — correctness_judge should appear")
print("Every new trace will now be automatically scored")

---
## Summary — What you built

```
MLflow-GenAI-Demo experiment
├── Overview       → 3 runs comparing temperatures
├── Traces         → every Gemini call recorded
├── Sessions       → 2 chat sessions (5 turns total)
├── Evaluation runs→ correctness + conciseness scores per answer
├── Datasets       → python-qa-v1 (3 rows), python-qa-v2 (5 rows)
└── Judges         → correctness_judge auto-scoring live traces
```

**In production** this same setup lets you:
- Compare model versions side by side (Overview)
- Debug exactly what the model received and returned (Traces)
- Track multi-turn conversations per user (Sessions)
- Measure quality on a test set before releasing (Evaluation runs)
- Version your test data so score changes are explainable (Datasets)
- Continuously monitor live traffic quality (Judges)